---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 2.3: Add More Tools

## 💬 Product Check-in: API Existence Checks

> **Sam (Product):** "Bigger issue — users are asking whether specific APIs exist in MLflow and the agent is confidently making things up. Someone asked about `mlflow.genai.evaluate` and the agent described a completely different function signature. They wasted an hour debugging code that was never going to work."
>
> **You:** "Yeah, that's a hallucination problem. The LLM has seen a lot of MLflow docs in training but it pattern-matches on what *sounds* right rather than what *is* right. It'll fabricate parameter names with complete confidence."
>
> **Sam:** "Can we do anything without a full doc retrieval system? That feels heavy for this sprint."
>
> **You:** "We can get most of the way there with a simple lookup tool. A hardcoded dictionary of the most commonly asked-about APIs — function name, parameters, a one-line description. The agent calls it instead of guessing. Not exhaustive, but it covers the high-traffic cases and stops the worst hallucinations."
>
> **Sam:** "How do we know it's using the lookup and not still guessing?"
>
> **You:** "Same as before — `ToolCallCorrectness` and `ToolCallEfficiency` in the eval pipeline. I'll also add eval cases where the expected tool call is explicitly specified so we can catch regressions."
>

---

Let's do exactly that. We'll:
1. Build the `check_api_exists` tool
2. Add it to the agent
3. Add `ToolCallCorrectness` and `ToolCallEfficiency` to the eval pipeline
4. Write eval cases with `expected_tool_calls` ground truth

```python
# These examples should FAIL before the tool is added —
# the bare LLM will hallucinate parameter names or confidently describe wrong signatures

```

In [ ]:
import importlib
import inspect


def check_api_exists(fully_qualified_name: str) -> dict:
    """
    Check whether an MLflow API exists and return its signature.

    Use this tool when a user asks whether a specific MLflow function, class,
    or method exists, or asks about its parameters. Pass the fully qualified
    name (e.g. "mlflow.genai.evaluate") and the tool will inspect the installed
    MLflow package directly — no guessing required.

    Args:
        fully_qualified_name: Dotted path to the API, e.g. "mlflow.set_experiment"

    Returns:
        A dict with 'exists', and if True, 'parameters' and 'description'.
    """
    parts = fully_qualified_name.rsplit(".", 1)
    if len(parts) != 2:
        return {"exists": False, "error": f"Invalid API path: {fully_qualified_name}"}

    module_path, attr_name = parts
    try:
        mod = importlib.import_module(module_path)
    except ModuleNotFoundError:
        return {"exists": False}

    obj = getattr(mod, attr_name, None)
    if obj is None:
        return {"exists": False}

    try:
        sig = inspect.signature(obj)
        parameters = list(sig.parameters.keys())
    except (ValueError, TypeError):
        parameters = []

    doc = inspect.getdoc(obj) or ""
    first_line = doc.split("\n")[0] if doc else "No description available."

    return {
        "exists": True,
        "name": fully_qualified_name,
        "parameters": parameters,
        "description": first_line,
    }

In [ ]:
# Quick smoke test
print(check_api_exists("mlflow.genai.evaluate"))
print(check_api_exists("mlflow.set_experiment"))
print(check_api_exists("mlflow.fake.nonexistent"))

In [ ]:
api_eval_dataset = [
    # ── Should call check_api_exists ───────────────────────────────────────
    {
        "inputs": {"question": "Does mlflow.genai.evaluate exist in MLflow?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "check_api_exists", "arguments": {"fully_qualified_name": "mlflow.genai.evaluate"}},
            ],
            "expected_facts": ["yes", "mlflow.genai.evaluate"],
            "guidelines": [
                "Response must confirm the API exists, not speculate",
                "Response must not fabricate parameter names not in the actual signature",
            ],
        },
    },
    {
        "inputs": {"question": "What parameters does mlflow.genai.evaluate take?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "check_api_exists", "arguments": {"fully_qualified_name": "mlflow.genai.evaluate"}},
            ],
            "expected_facts": ["data", "scorers"],
            "guidelines": [
                "Response must only reference parameter names that exist in the actual API",
                "Response must not describe the pre-3.0 mlflow.evaluate() signature",
            ],
        },
    },
    {
        "inputs": {"question": "Is there a Guidelines scorer in MLflow?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "check_api_exists", "arguments": {"fully_qualified_name": "mlflow.genai.scorers.Guidelines"}},
            ],
            "expected_facts": ["yes", "Guidelines"],
            "guidelines": [
                "Response must confirm the scorer exists rather than guessing",
                "Response must not conflate it with a different MLflow API",
            ],
        },
    },
    {
        "inputs": {"question": "Does mlflow.genai.magic_optimize exist?"},
        "expectations": {
            "expected_tool_calls": [
                {"name": "check_api_exists", "arguments": {"fully_qualified_name": "mlflow.genai.magic_optimize"}},
            ],
            "expected_facts": ["no"],
            "guidelines": [
                "Response must clearly state the API does not exist",
                "Response must not hallucinate a description of a nonexistent function",
            ],
        },
    },
    # ── Should NOT call check_api_exists ───────────────────────────────────
    {
        "inputs": {"question": "What's the best way to chop an onion?"},
        "expectations": {
            "expected_tool_calls": [],
        },
    },
    {
        "inputs": {"question": "How do I make a good pasta sauce?"},
        "expectations": {
            "expected_tool_calls": [],
            "expected_facts": ["tomato", "garlic"],
        },
    },
]
print(f"API eval set size: {len(api_eval_dataset)} examples")
print(f"  - should call tool: {sum(1 for e in api_eval_dataset if e['expectations'].get('expected_tool_calls'))}")
print(f"  - should NOT call tool: {sum(1 for e in api_eval_dataset if e['expectations'].get('expected_tool_calls') == [])}")